In [1]:
import pandas as pd 
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os 

In [2]:
load_dotenv()

USER = os.getenv('DB_USER')
PASS = os.getenv('DB_PASS')
HOST = os.getenv('DB_HOST')
PORT = os.getenv('DB_PORT')
NAME = os.getenv('DB_NAME')

engine = create_engine(f'mysql+pymysql://{USER}:{PASS}@{HOST}:{PORT}/{NAME}')
print(' Bridge to MySQL ready to use')


 Bridge to MySQL ready to use


# Function to convert DF to CSV
CSV for tableau


In [ ]:

def converting_to_csv(df,file_name):
    '''
    Universal function to convert mutiple DataFrame to csv format.
    automaticly detect/make folders, prevent/minimize errors
    '''
    folder_name = 'Analysis Result'
    try:
        if not file_name.endswith('.csv'):
            file_name = file_name +'.csv'

        os.makedirs(folder_name, exist_ok= True)

        saved_path =os.path.join(folder_name, file_name)
        df.to_csv(saved_path, index= False)

    except Exception as e:
        print(f'failed to convert/save file. Error:{e}')


# Bussiness Question
1. What is the total revenue generated month-over-month?
2. Which geographic regions experience the highest freight costs and delivery delays?
3. Who are the top 5% most valuable customers based on total spending?

## Montlhy Revenue

In [10]:
monthly_query ='''
    select date_format(o.order_purchase_timestamp, '%%Y-%%m') as Month, sum(p.payment_value) as total_revenue
    from orders o join payment p
    on o.order_id = p.order_id
    group by Month 
    order by Month asc
'''
df_Monthly = pd.read_sql(monthly_query, con = engine)
df_Monthly['MoM_growth_percent'] = df_Monthly['total_revenue'].pct_change() * 100 
df_format_Monthly = df_Monthly.style.format({'total_revenue': '{:,.2f}'})
df_format_Monthly


,Month,total_revenue,MoM_growth_percent
0,2016-10,"47,271.20",nan
1,2016-12,19.62,-99.958495
2,2017-01,"127,545.67",649979.867482
3,2017-02,"271,298.65",112.707064
4,2017-03,"414,369.39",52.735515
5,2017-04,"390,952.18",-5.651289
6,2017-05,"566,872.73",44.997971
7,2017-06,"490,225.60",-13.521047
8,2017-07,"566,403.93",15.539443
9,2017-08,"646,000.61",14.052989


In [11]:
converting_to_csv(df_Monthly,"Monthly Revenue.csv")

## Highest freight cost and delivery delays regions
Regions with highest freight cost and delivery delays

In [17]:
Regions_query = '''
select 
    c.customer_state as Region, 
    avg(oi.freight_value) as average_freight, 
    count(distinct o.order_id) as sum_of_delay,
    avg(datediff(o.order_delivered_customer_date, o.order_estimated_delivery_date)) as average_delay
from 
    customers c join orders o
    on c.customer_id = o.customer_id 
    join order_items oi 
    on o.order_id = oi.order_id
where 
    datediff(o.order_delivered_customer_date, o.order_estimated_delivery_date) > 0
group by c.customer_state
order by 
    average_delay desc,
    sum_of_delay desc
'''
df_regions = pd.read_sql(Regions_query, con = engine)
df_regions

,Region,average_freight,sum_of_delay,average_delay
0,AP,28.086667,2,96.3333
1,RR,33.100000,5,36.4000
2,AM,55.226000,4,24.4000
3,AC,57.453333,3,18.6667
4,CE,32.548247,176,14.6959
5,SE,40.152459,51,14.5738
6,RN,35.754043,44,14.3404
7,RJ,21.963054,1495,13.2555
8,PI,36.623099,66,12.7606
9,PA,36.731597,106,12.5210


In [18]:
converting_to_csv(df_regions, "Highest Freight cost and delays region.csv")

## Most valuable customers
Top 5% Most Valuable customers based on total spendings

In [28]:
customer_query = '''
WITH RankedCustomers AS (
    SELECT
        c.customer_unique_id AS customer,
        SUM(p.payment_value) AS Total_spendings,
        NTILE(20) OVER (ORDER BY SUM(p.payment_value) DESC) AS percentile_group
    FROM
        customers c 
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN payment p ON o.order_id = p.order_id
    GROUP BY
        c.customer_unique_id
)
SELECT 
    customer,
    Total_spendings
FROM 
    RankedCustomers
WHERE 
    percentile_group = 1; 
'''
df_customers = pd.read_sql(customer_query, con= engine)
df_customers

,customer,Total_spendings
0,0a0a92112bd4c708ca5fde585afaa872,13664.08
1,da122df9eeddfedc1dc1f5349a1a690c,7571.63
2,763c8b1c9c68a0229c42c9fc6f662b93,7274.88
3,dc4802a71eae9be1dd28f5d788ceb526,6929.31
4,459bef486812aa25204be022145caa62,6922.21
...,...,...
4663,1fe7afd58ce8429c5a8d27bf8c24738f,469.89
4664,edaf10f5aa825cb0f5fb2c0e56c45891,469.89
4665,06e064e7008f3d351d74a80da7576bcc,469.84
4666,a3273cc5b69f034bf090c924b75c725d,469.78


In [30]:
converting_to_csv(df_customers, "Top 5% Customers based on Total spendings.csv")

# Data Ready for Visualization in Tableau